# Projet d'autonomie : cartes postales anciennes (détection et transcription)

**Pool avancé du M1 Sciences des Données, en parallèle du cours.**

Vous faites déjà du Python depuis un moment : ce projet est une piste bonus pour ne
pas vous ennuyer pendant que le reste du groupe consolide les bases. On vous confie un
**vrai** jeu de données de recherche et une **vraie** tâche difficile et ouverte. Le but
n'est pas de suivre une recette, mais d'apprendre à vous débrouiller : chercher, lire,
tester, décider, comparer. Vous êtes en master, c'est le moment. Si vous pensez que la tâche 
est trop compliquée pour votre niveau, voir la section **Repli facultatif**.

### Les règles du jeu

- **Livrable :** ce notebook, complété par vos soins. Vous le rendez sous forme d'un **lien vers votre dépôt** ou d'une **archive**, avec de quoi reproduire votre environnement (voir la section « Rendu »).
- **Pas de corrigé.** Personne ne vous donnera la solution pas-à-pas. Les objectifs, les
  jalons et quelques indices repliables sont là pour vous cadrer, pas pour vous tenir la main.
- **Autonomie de recherche.** À vous de trouver les bons outils et de lire leur documentation.
  On ne vous impose aucune bibliothèque ni aucun tutoriel : c'est une compétence à travailler.
- **Liberté.** Amusez-vous, explorez au-delà des deux tâches si l'envie vient.
- **Cible.** On vous donne, dans les sections concernées, les **scores de référence** obtenus
  par l'enseignant. Ils servent de **cible à approcher, puis à battre**. On ne vous dit pas
  avec quels modèles ni quelles bibliothèques ils ont été obtenus : à vous de trouver comment
  vous en rapprocher.

### Les indices repliables

Tout au long du notebook, des indices sont cachés dans des blocs dépliables. Ouvrez-les
seulement si vous bloquez :

<details>
<summary>Exemple d'indice (cliquez pour ouvrir)</summary>

Voilà à quoi ressemble un indice. Il vous oriente sans vous donner la réponse.
Essayez toujours par vous-même d'abord.

</details>


## Le jeu de données

**Historical Postcards Dataset**, publié sur Recherche Data Gouv.

- **DOI :** [10.57745/GELGHH](https://doi.org/10.57745/GELGHH)
- **Licence du contenu :** Licence Ouverte Etalab 2.0 (`etalab-2.0`). Le dataset est donc
  librement réutilisable, y compris pour ce projet.
- **Format :** COCO. Des images (scans de cartes postales anciennes) et des fichiers
  d'annotations JSON `instances_<Set>.json`, où `<Set>` vaut `Train`, `Test` ou `Synth`.
  Les annotations portent des **attributs personnalisés** (dont, pour les zones de texte,
  la **transcription**).

### Se procurer les données

Téléchargez le dataset depuis le DOI ci-dessus (page Recherche Data Gouv). Les scores de
référence de ce sujet ont été mesurés sur le **set de Test**, c'est donc le point de départ
naturel pour évaluer votre travail. Utilisez le set `Train` pour l'entraînement, mais rien ne vous empêche d'utiliser aussi le set synthétique (`Synth`) si votre approche en a besoin (attention, la totalité du jeu de données pèse plusieurs Go).

Une fois décompressé, vous obtenez (l'arborescence exacte peut varier légèrement) :

```
dataset/
├─ annotations/instances_Test.json   # annotations COCO du set de Test
└─ images/Test/*.jpg                 # les images correspondantes
```

Le champ `file_name` de chaque image dans le JSON vous indique le nom de fichier attendu :
adaptez au besoin le chemin des images dans la cellule de configuration plus bas.

### Citation

> Pélingre, M. & Tabbone, S. A. (2026). *Historical Postcards Dataset (COCO), v2.0.*
> Recherche Data Gouv. DOI : [10.57745/GELGHH](https://doi.org/10.57745/GELGHH).
> Contenu sous Licence Ouverte Etalab 2.0.

Merci de conserver cette citation et la mention de licence dans votre rendu.


## La carte des jalons

Une progression indicative, du plus simple au plus ouvert. Vous n'êtes pas obligés de tout
faire : allez aussi loin que vous le pouvez, et documentez vos choix.

1. **Prise en main :** ouvrir les annotations COCO, explorer (combien d'images ? de
   catégories ? d'annotations par image ? quels attributs ?).
2. **EDA et visualisation :** distributions (catégories, tailles de boîtes, dimensions
   d'images), affichage des annotations superposées aux images, statistiques sur les
   transcriptions.
3. **Détection :** mettre en place puis évaluer un détecteur de zones, et comparer aux
   scores de référence.
4. **Transcription :** appliquer une reconnaissance de texte (imprimé et manuscrit) sur la totalité des images ou sur les
   régions, mesurer la qualité, comparer aux scores de référence, en distinguant les types
   de texte si possible.
5. **Stretch (libre) :** tout ce qui vous inspire (amélioration, comparaison de modèles,
   analyse spatiale, NLP sur les transcriptions...).

Un **repli facultatif** vers un jeu de données plus simple est proposé à la fin, au cas où
la tâche vous paraîtrait trop dure au démarrage.


## 0. Environnement : uv (obligatoire pour une utilisation en local)

**Point de départ imposé :** votre environnement doit être géré dans un environnement virtuel avec
[uv](https://docs.astral.sh/uv/), pour qu'il soit **reproductible**. Pas d'installation
sauvage en `pip install` global : tout passe par un projet uv.

Dans un terminal, initialisez le projet et lancez Jupyter **depuis uv** (c'est ce qui garantit
que le noyau du notebook voit bien les paquets installés) :

```bash
# 1. Créer le projet et s'y placer
uv init postcards-projet
cd postcards-projet

# 2. Ajouter la base neutre (lecture COCO, images, tracés)
uv add jupyterlab numpy pillow matplotlib pycocotools

# 3. Lancer Jupyter dans l'environnement du projet, puis ouvrir ce notebook
uv run jupyter lab
```

> NB. vous pouvez choisir la python voulue lors de l'initialisation à l'aide `--python`. Exemple avec la 3.12 : `uv init --python 3.12 postcards-projet`.  
> Vous avez oublié de le faire ou vous voulez changer de version ? Pas de panique : `uv python pin 3.13` puis `uv sync` pour passer à la 3.13 par exemple !

Ensuite, **à vous d'ajouter** les briques dont vous aurez besoin pour la détection et pour
la reconnaissance de texte, au fur et à mesure de vos choix :

```bash
uv add <la-bibliotheque-que-vous-avez-choisie>
```

On ne vous dit pas lesquelles : c'est justement une partie du travail.

<details>
<summary>Encadré Windows</summary>

`uv` fonctionne aussi sous Windows (PowerShell). L'installation se fait via l'installeur
officiel ou `pipx`/`pip`. Les commandes ci-dessus sont identiques. Attention aux chemins :
utilisez `pathlib.Path` (voir la cellule de configuration) plutôt que des chaînes avec des
antislash, pour que votre code reste portable.

</details>

<details>
<summary>Indice : garder l'environnement traçable dans le notebook</summary>

`uv` écrit un `pyproject.toml` et un `uv.lock` : ce sont eux qui rendent votre environnement
reproductible. Vous pouvez les joindre à votre rendu. Pour une variante « un seul fichier »,
regardez du côté des métadonnées de script inline (PEP 723) et de l'exécution `uv run`.

</details>


### Variante : Google Colab (si votre machine n'est pas assez puissante)

Pas de GPU, pas assez de mémoire, ou une installation locale qui résiste ? Vous pouvez mener
tout le projet sur [Google Colab](https://colab.research.google.com/), un environnement en ligne
qui donne accès à un GPU gratuit (bien utile pour la détection et la reconnaissance de texte).

Marche à suivre :

1. Ouvrez [Google Colab](https://colab.research.google.com/), menu *Fichier > Importer un
   notebook*, et déposez ce fichier `.ipynb`.
2. Activez le GPU : menu *Exécution > Modifier le type d'exécution > Accélérateur matériel : GPU*.
3. Colab fournit déjà un environnement Python prêt à l'emploi : ici, on **n'utilise pas uv**, on
   installe simplement avec `pip` (cellule ci-dessous). La discipline uv (projet, `pyproject.toml`,
   `uv.lock`, `.python-version`) reste pour le travail **en local**.
4. Ajoutez de la même manière les briques de détection et d'OCR que vous choisirez :
   `!pip install <votre-bibliotheque>`.

Deux points d'attention propres à Colab :

- **Le runtime est éphémère** : à chaque nouvelle session, il faut relancer l'installation. Pour
  la reproductibilité et le rendu, figez vos dépendances dans un `requirements.txt`
  (`!pip freeze > requirements.txt`, puis téléchargez-le) et notez la version de Python de Colab
  (`!python --version`). Voir la section « Rendu ».
- **Les données** : le jeu complet pèse plusieurs Go. Le plus pratique est de le déposer sur
  votre Google Drive et de le monter dans Colab
  (`from google.colab import drive; drive.mount('/content/drive')`), puis de faire pointer
  `DATA_DIR` vers le bon dossier. Vous pouvez aussi ne travailler que sur le set de Test pour
  démarrer.

<details>
<summary>Indice : garder la main sur les versions sur Colab</summary>

Colab arrive avec ses propres versions de nombreux paquets. Si un conflit apparaît, épinglez
la version dont vous avez besoin (`!pip install "paquet==x.y.z"`) et gardez-en la trace. Un
`!pip freeze` en fin de session vous donne l'instantané complet à joindre au rendu.

</details>


In [ ]:
import sys
import subprocess

# À exécuter sur Google Colab uniquement : Colab fournit déjà un environnement Python,
# on installe donc simplement avec pip (uv reste pour le travail en local).
if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "numpy", "pillow", "matplotlib", "pycocotools"],
        check=True,
    )
    print("Base installée sur Colab via pip.")
    print("Ajoutez vos briques avec, par exemple :  !pip install <votre-bibliotheque>")
    print("En fin de session, figez l'environnement :  !pip freeze > requirements.txt")
else:
    print("Hors Colab : rien à faire ici. Installez via uv en local (voir la section ci-dessus).")


La cellule suivante **vérifie** que la base est bien en place. Si elle signale des
paquets manquants, c'est que le notebook n'a pas été lancé depuis l'environnement uv
en local (ou, sur Colab, que la cellule d'installation ci-dessus n'a pas été exécutée) :
revenez aux étapes correspondantes ci-dessus.

In [ ]:
import importlib


def verifier_environnement():
    """Vérifie que les paquets de base du projet sont importables et affiche leurs versions.

    Renvoie True si tous les paquets de base sont présents, False sinon (avec un message
    indiquant quoi faire).
    """
    a_verifier = [
        ("numpy", "numpy"),
        ("PIL", "pillow"),
        ("matplotlib", "matplotlib"),
        ("pycocotools", "pycocotools"),
    ]
    versions = {}
    manquants = []
    for nom_import, nom_paquet in a_verifier:
        try:
            module = importlib.import_module(nom_import)
            versions[nom_paquet] = getattr(module, "__version__", "?")
        except ImportError:
            manquants.append(nom_paquet)
    tout_ok = (len(manquants) == 0)
    if tout_ok:
        print("Environnement de base prêt. Versions détectées :")
        for paquet, version in versions.items():
            print(f"  - {paquet}: {version}")
    else:
        print("Paquets de base manquants :", ", ".join(manquants))
        print("Lancez ce notebook depuis l'environnement uv (voir la section ci-dessus),")
        print("puis ajoutez ce qui manque avec :  uv add " + " ".join(manquants))
    return tout_ok


verifier_environnement()


Configuration des chemins. **Adaptez-les** à l'endroit où vous avez décompressé le
dataset. On utilise `pathlib` pour rester portable (Windows comme Linux).

In [ ]:
from pathlib import Path

# À adapter : dossier racine où vous avez décompressé le dataset (voir "Se procurer les données").
DATA_DIR = Path("dataset")
ANN_FILE = DATA_DIR / "annotations" / "instances_Test.json"      # annotations COCO du set de Test
IMAGES_DIR = DATA_DIR / "images" / "Test"          # images correspondantes (à ajuster si besoin)

print("Annotations :", ANN_FILE, "->", "trouvé" if ANN_FILE.exists() else "ABSENT (à télécharger / chemin à corriger)")
print("Images      :", IMAGES_DIR, "->", "trouvé" if IMAGES_DIR.exists() else "ABSENT (à télécharger / chemin à corriger)")


## 1. Prise en main : charger et explorer les annotations

**Objectif.** Ouvrir les annotations COCO et vous faire une idée précise de ce qu'il y a
dedans. C'est la fondation de tout le reste : ne la bâclez pas.

**Jalons.**

- [ ] Charger le fichier d'annotations et repérer sa structure de haut niveau.
- [ ] Compter les images et les annotations.
- [ ] Lister les **catégories** (et leurs éventuelles supercatégories).
- [ ] Repérer les **attributs** portés par les annotations, et lesquels concernent le texte.
- [ ] Compter les annotations par catégorie, et les annotations par image.

**À vous.** Écrivez le code d'exploration dans la cellule ci-dessous. Formulez à voix haute
(en commentaire ou en markdown) ce que vous découvrez : c'est ce qui guidera vos choix plus tard.

<details>
<summary>Indice : comment lire du COCO</summary>

Un fichier COCO est un simple JSON : vous pouvez l'ouvrir avec le module standard `json` et
inspecter ses clés de premier niveau. Il existe aussi une bibliothèque dédiée (installée dans
la base) qui expose des méthodes pratiques pour naviguer entre images, annotations et
catégories, et qui resservira pour l'évaluation de la détection. À vous de voir laquelle des
deux voies vous convient.

</details>

<details>
<summary>Indice : où vivent les transcriptions ?</summary>

Toutes les annotations ne se ressemblent pas. Regardez le champ `attributes` d'une annotation :
certaines catégories le remplissent, d'autres non. Pour les zones de texte, vous y trouverez
notamment la transcription et des informations comme la langue, l'orientation, ou un marqueur
d'illisibilité. Les noms exacts des champs se découvrent en inspectant quelques annotations.

</details>


In [ ]:
# À vous de jouer : chargez ANN_FILE et explorez.
# Pistes : structure de haut niveau, nombre d'images / d'annotations,
#          catégories, attributs des annotations, comptages par catégorie et par image.


## 2. Exploration et visualisation

**Objectif.** Comprendre le jeu de données visuellement et statistiquement. Une bonne EDA
révèle les difficultés (petites boîtes, textes tournés, illisibles, déséquilibre des classes)
avant même de modéliser.

**Jalons.**

- [ ] Distribution des annotations par **catégorie** (le jeu est-il équilibré ?).
- [ ] Distribution des **tailles de boîtes** (largeur, hauteur, aire) et des **dimensions
      d'images**.
- [ ] **Afficher** quelques images avec leurs annotations superposées (boîtes + libellé de
      catégorie, voire la transcription).
- [ ] Statistiques sur les **transcriptions** : longueurs, langues présentes, proportion de
      zones marquées illisibles, présence de retours à la ligne.

**À vous.** Produisez au moins trois visualisations qui vous aident à décider de votre stratégie.

<details>
<summary>Indice : dessiner une boîte COCO sur une image</summary>

En COCO, une boîte `bbox` est au format `[x, y, largeur, hauteur]` en pixels, l'origine en
haut à gauche. Pour la tracer par-dessus une image, un rectangle d'axes (par exemple un
`Rectangle` de matplotlib) fait l'affaire. Attention : les scans sont **grands** (plusieurs
milliers de pixels de côté) ; pensez à redimensionner l'affichage ou à ne montrer qu'un
extrait pour que ce soit lisible et rapide.

</details>

<details>
<summary>Indice : petites boîtes de texte</summary>

Les zones de texte sont souvent larges et peu hautes (des lignes). Regardez la distribution
des hauteurs de boîtes : elle vous dira si un modèle générique de détection d'objets aura du
mal, et si un pré-traitement (recadrage, mise à l'échelle) sera utile pour l'étape de
transcription.

</details>


In [ ]:
# À vous de jouer : EDA et visualisations.
# Au moins trois figures utiles (distributions, tailles, image annotée...).


## 3. Détection des zones

**Objectif.** Mettre en place un détecteur qui retrouve les zones annotées (les six catégories),
puis l'**évaluer** sur le set de Test et comparer aux scores de référence. Distinguez le cas
« toutes zones confondues » et le cas « texte imprimé seul », qui est plus facile.

**Comment lire la cible.** Les métriques sont les métriques de détection COCO usuelles :

- **Précision** = proportion de vos détections qui sont correctes.
- **Rappel** = proportion des zones réelles que vous retrouvez.
- **mAP@50** = *mean Average Precision* à un seuil de recouvrement (IoU) de 0,50 : la métrique
  de référence en détection. Plus c'est haut, mieux c'est, pour les trois colonnes.

Une détection est comptée correcte si son recouvrement (IoU) avec une boîte réelle de la bonne
catégorie dépasse 0,50.

**Scores de référence (cible à approcher, puis à battre).**

| Portée | Précision | Rappel | mAP@50 |
|---|---|---|---|
| Toutes zones | 0,83 | 0,82 | 0,81 |
| Texte imprimé seul | 0,900 | 0,987 | 0,985 |

**Jalons.**

- [ ] Choisir une stratégie de détection (modèle pré-entraîné à réutiliser, à ré-entraîner,
      à spécialiser... c'est votre décision).
- [ ] Produire des prédictions sur le set de Test au format exploitable pour l'évaluation.
- [ ] Calculer précision, rappel et mAP@50, globalement puis pour la seule classe de texte
      imprimé.
- [ ] Vous situer par rapport à la cible et analyser vos erreurs (quelles catégories coincent ?).

**À vous.** On ne vous dit pas quel détecteur employer, ni quelle bibliothèque. Cherchez,
comparez, justifiez votre choix.

<details>
<summary>Indice : évaluer proprement en COCO</summary>

La bibliothèque de lecture COCO de la base fournit aussi un module d'évaluation qui calcule
précision, rappel et AP à différents seuils d'IoU une fois que vos prédictions sont au bon
format (une liste d'objets avec `image_id`, `category_id`, `bbox` et un score de confiance).
Vous pouvez aussi coder votre propre appariement par IoU pour bien comprendre ce que mesurent
ces chiffres. Le format des prédictions et le calcul de l'IoU sont des points classiques à ne
pas négliger.

</details>

<details>
<summary>Indice : la classe « texte imprimé » est plus facile</summary>

La ligne « texte imprimé seul » de la cible est nettement plus haute que la ligne globale.
C'est un bon point d'entrée : commencez peut-être par bien détecter cette classe avant de
vous attaquer aux zones plus rares ou plus irrégulières.

</details>


In [ ]:
# À vous de jouer : mise en place et évaluation d'un détecteur.
# Objectif : précision, rappel, mAP@50 (global et texte imprimé) sur le set de Test.


## 4. Transcription du texte

**Objectif.** Sur les régions de texte, produire la transcription automatique et la comparer
au texte réel (l'attribut de transcription des annotations). Mesurez la qualité et comparez à
la cible, **en distinguant les trois types de texte** : imprimé, manuscrit, et texte « naturel »
(le texte présent dans la scène de la carte, enseignes, panneaux...). Ces trois types n'ont pas
du tout la même difficulté.

**Comment lire la cible.** Trois mesures, deux familles :

- **Similarité de Levenshtein (normalisée)** : 1 moins la distance d'édition rapportée à la
  longueur. Vaut 1 pour une transcription parfaite. **Plus c'est haut, mieux c'est.**
- **Jaccard (mots)** : recouvrement des **ensembles de mots** (intersection sur union) entre
  la transcription prédite et la transcription réelle. **Plus c'est haut, mieux c'est.**
- **CER** (*Character Error Rate*) : taux d'erreur par caractère, la distance d'édition
  rapportée au nombre de caractères de référence. **Plus c'est bas, mieux c'est.**

**Scores de référence (cible).** Flèche ↑ : plus haut = mieux. Flèche ↓ : plus bas = mieux.

| Type de texte | Similarité Levenshtein (norm.) ↑ | Jaccard mots ↑ | CER ↓ |
|---|---|---|---|
| Imprimé | 0,976 | 0,949 | 0,034 |
| Manuscrit | 0,775 | 0,476 | 0,238 |
| Texte naturel | 0,850 | 0,713 | 0,189 |

L'imprimé est presque « résolu » ; le manuscrit est le plus dur. Ne soyez pas surpris que vos
scores manuscrits soient bien plus bas que vos scores imprimés : c'est attendu.

**Jalons.**

- [ ] Récupérer les régions de texte (recadrées depuis les images grâce aux boîtes) et leur
      transcription de référence.
- [ ] Choisir et appliquer un moteur de reconnaissance de texte (imprimé et/ou manuscrit).
- [ ] Implémenter (ou réutiliser) les trois mesures ci-dessus et les calculer **par type de
      texte**.
- [ ] Vous situer par rapport à la cible et discuter : où est la marge de progrès ? Que faire
      des zones illisibles, des textes tournés, des langues différentes ?

**À vous.** Là encore, le moteur de reconnaissance et les bibliothèques sont votre choix. Vous pouvez également essayer de transcrire les textes sur la carte entière, sans détecteur.

<details>
<summary>Indice : préparer les régions</summary>

Chaque boîte de texte peut être recadrée dans l'image d'origine pour ne donner au moteur de
reconnaissance que la zone utile. Deux détails qui comptent : certaines zones sont marquées
comme illisibles (leur transcription de référence est vide, décidez si vous les gardez ou non),
et certaines ont une orientation (0, 90, 180, 270 degrés) qu'il peut valoir la peine de
corriger avant reconnaissance.

</details>

<details>
<summary>Indice : calculer les mesures</summary>

La **distance de Levenshtein** (distance d'édition) est la brique commune : la similarité
normalisée vaut `1 - distance / max(len(pred), len(vrai))`, et le CER vaut
`distance / len(vrai)`. Le **Jaccard mots** se calcule sur les ensembles de mots :
`|A ∩ B| / |A ∪ B|`. Pensez à normaliser les textes de la même façon des deux côtés (casse,
espaces, ponctuation) et à décider comment vous agrégez : moyenne par annotation, puis moyenne
par type de texte. Coder la distance d'édition vous-mêmes est un bon exercice ; une
bibliothèque fait aussi l'affaire.

</details>


In [ ]:
# À vous de jouer : transcription des régions et mesures par type de texte.
# Objectif : similarité de Levenshtein normalisée, Jaccard mots, CER, par type (imprimé / manuscrit / naturel).


## 5. Pour aller plus loin (libre)

Si vous avez de l'avance ou de la curiosité, quelques directions (non exhaustives, non
imposées) :

- Comparer plusieurs modèles de détection ou de reconnaissance et discuter le compromis
  qualité / vitesse / mémoire.
- Améliorer les cas durs : manuscrit, textes tournés, petites boîtes, zones illisibles.
- Analyse spatiale : où se trouvent les timbres, les cachets, le texte, sur la carte ?
- NLP sur les transcriptions : langues, entités nommées (lieux, dates), nettoyage OCR.
- Robustesse : que donne votre pipeline sur des cartes très abîmées ?

Documentez vos expériences, même celles qui échouent : c'est ça, la démarche de recherche.


## Repli facultatif : un jeu de données plus simple

Cette tâche est volontairement difficile. **Si vous bloquez au démarrage**, ne restez pas
coincés : basculez sur un jeu de données **plus simple, de votre choix**, sur Kaggle (ou
ailleurs), et gardez la **même ossature** de projet (chargement, EDA, une tâche de détection
**ou** de reconnaissance de texte, évaluation avec des métriques claires).

Le dataset n'est **pas imposé** : prenez celui qui vous parle. Un bon repli, c'est typiquement
un corpus de reconnaissance de texte imprimé « propre », ou un petit jeu de détection d'objets
bien balisé, avec assez d'exemples pour évaluer sérieusement. L'important est de pratiquer la
démarche complète de bout en bout, quitte à revenir ensuite aux cartes postales.

Vous pouvez même partir sur d'autres tâches : analyses de séries temporelles, classification, régression.
S'il vous manque des compétences en machine learning, je vous conseille plutôt de suivre le mooc : [Machine learning in Python with scikit-learn](https://www.fun-mooc.fr/fr/cours/machine-learning-python-scikit-learn/)

<details>
<summary>Indice : ce qui rend un jeu « plus simple »</summary>

Cherchez : images nettes et cadrées, classes peu nombreuses et équilibrées, annotations déjà
propres, volume raisonnable, et une métrique d'évaluation standard bien documentée. Vous
gagnerez du temps sur la logistique et pourrez vous concentrer sur la modélisation.

</details>


## Chercher par vous-mêmes

On ne vous fournit pas de liste de ressources : savoir se documenter fait partie du travail.
Quelques réflexes utiles : lire la documentation officielle des outils que vous choisissez,
regarder comment d'autres évaluent la détection COCO et la reconnaissance de texte, et, si vous
voulez creuser le contexte du dataset, la page du DOI ([10.57745/GELGHH](https://doi.org/10.57745/GELGHH))
donne accès à sa documentation et à des publications associées. À vous de décider ce que vous
lisez et utilisez.

## Rendu

Deux formats au choix : un **lien vers le dépôt** de votre projet, ou une **archive** regroupant
les fichiers demandés.

- [ ] Ce notebook, complété (code, figures, et vos commentaires / décisions).
- [ ] Vos scores, comparés à la cible, avec une brève analyse.
- [ ] De quoi **reproduire votre environnement** :
    - en local (uv) : `pyproject.toml`, `uv.lock` **et** `.python-version` ;
    - sur Colab (pip) : un `requirements.txt` (via `pip freeze`) **et** la version de Python
      utilisée (sortie de `python --version`).
- [ ] La citation du dataset et la mention de licence conservées.

**Citation du dataset**

> Pélingre, M. & Tabbone, S. A. (2026). *Historical Postcards Dataset (COCO), v2.0.*
> Recherche Data Gouv. DOI : [10.57745/GELGHH](https://doi.org/10.57745/GELGHH).
> Contenu sous **Licence Ouverte Etalab 2.0**.

Bon courage, et amusez-vous.
